# TransferGuard — CIFAR-10, 40% source-label noise

**Local NVIDIA GPU | Clean follow-up experiment**

This notebook tests whether a **partially unreliable source CNN** helps or hurts clean CIFAR-10 target learning. It follows the model, split, SGD, and dynamic-label-noise setup of your previous 80%-noise notebook, changing only the source noise rate to **40%**.

**We train:**

1. **Source BigCNN:** 10,000 source images with 40% *dynamically resampled* wrong labels (60 epochs).
2. **Scratch BigCNN:** 4,000 clean target images (30 epochs). Reuse your earlier checkpoint if the data split matches.
3. **Frozen transfer:** load the 40%-noise source, freeze all but a freshly reset final classifier (30 epochs).
4. **Full fine-tuning:** load the source, reset only the final classifier, train all layers (30 epochs).
5. **Reset-both fine-tuning:** load the source, reset both FC layers, train all layers (30 epochs).

The output is **validation accuracy only**. The 500-image calibration subset is reserved for the future TransferGuard detector, and the official CIFAR-10 test set is deliberately *not evaluated here*.

**Reference:** Xu et al., *Residual Feature Integration Is Sufficient to Prevent Negative Transfer* (ICLR 2026), [paper](https://arxiv.org/abs/2505.11771) and [code](https://github.com/Xyc-arch/ReFine-residual-feature-integration). This notebook is an **adapted pilot**, not an exact numerical reproduction. No negative-transfer result is guaranteed.

## 1. Setup

Select the **CUDA-enabled Python kernel** in VS Code/Jupyter. The required packages are `torch`, `torchvision`, `numpy`, `pandas`, and `matplotlib`. If your existing TransferGuard GPU environment works, no reinstall is needed.

In [ ]:
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import datasets, transforms

seed = 42
noise_rate = 0.40
source_epochs = 60
target_epochs = 30
batch_size = 16
learning_rate = 0.01

root = Path.cwd().resolve()
if root.name == 'notebooks':
    root = root.parent
if not (root / 'notebooks').is_dir():
    raise RuntimeError('Start Jupyter from the repository root or notebooks directory.')
folder = root / 'transferguard_noise40_results'
folder.mkdir(exist_ok=True)
data_dir = root / 'data'
data_dir.mkdir(exist_ok=True)


def set_seed(value):
    random.seed(value)
    np.random.seed(value)
    torch.manual_seed(value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(value)


set_seed(seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type != 'cuda':
    raise RuntimeError('CUDA not found. Select your existing GPU-enabled Python kernel.')

print('GPU:', torch.cuda.get_device_name(0))
print('Results:', folder.resolve())

## 2. Dataset and fixed splits

The code uses the **same seed and index ranges** as your previous 80% notebook. Data subsets never overlap. The source training labels are noisy; target, validation, and calibration labels stay clean.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_data = datasets.CIFAR10(
    root=str(data_dir), train=True, download=True, transform=transform
)

indices = np.random.RandomState(seed).permutation(len(train_data))
source_ids = indices[:10000]
target_ids = indices[10000:14000]
val_ids = indices[14000:16000]
cal_ids = indices[16000:16500]

assert len(set(source_ids) | set(target_ids) | set(val_ids) | set(cal_ids)) == 16500
np.savez(folder / 'split_indices_seed42.npz', source=source_ids,
         target=target_ids, validation=val_ids, calibration=cal_ids)

source_clean = Subset(train_data, source_ids)
target_data = Subset(train_data, target_ids)
val_data = Subset(train_data, val_ids)
cal_data = Subset(train_data, cal_ids)


class NoisyLabels(Dataset):
    def __init__(self, data, rate):
        self.data = data
        self.rate = rate

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        image, label = self.data[index]
        if np.random.rand() < self.rate:
            wrong = np.random.randint(0, 9)
            if wrong >= label:
                wrong += 1
            label = int(wrong)
        return image, label


def make_loader(data, shuffle=False):
    return DataLoader(data, batch_size=batch_size, shuffle=shuffle,
                      num_workers=0, pin_memory=True)


source_loader = make_loader(NoisyLabels(source_clean, noise_rate), shuffle=True)
target_loader = make_loader(target_data, shuffle=True)
val_loader = make_loader(val_data)

print('Source:', len(source_clean), '| Target:', len(target_data),
      '| Validation:', len(val_data), '| Reserved calibration:', len(cal_data))
print('Source label noise:', int(noise_rate * 100), '% (resampled on every access)')

## 3. BigCNN

**Same architecture as your earlier experiment.** Keeping it unchanged makes the 40%- and 80%-source conditions more comparable.

In [ ]:
class BigCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 80, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(80, 160, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(160, 320, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(320, 640, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(640, 640, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(640, 768, 3, padding=1), nn.ReLU()
        )
        self.fc_layers = nn.Sequential(
            nn.Linear(768, 2560), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(2560, 10)
        )

    def forward(self, x):
        return self.fc_layers(self.conv_layers(x).flatten(1))


print('BigCNN parameters:', sum(p.numel() for p in BigCNN().parameters()))

## 4. Training functions

One function is shared by every run: SGD with learning rate 0.01 and momentum 0.9, best-validation checkpoint, per-epoch CSV and resumable `_last.pt`. A run resumes automatically when the matching file exists. Use a **new run name** if you change a model's initialization or settings.

In [ ]:
@torch.no_grad()
def accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(1) == y).sum().item()
        total += len(y)
    return 100 * correct / total


def train_model(model, loader, name, epochs):
    model = model.to(device)
    optimizer = torch.optim.SGD(
        [p for p in model.parameters() if p.requires_grad],
        lr=learning_rate, momentum=0.9
    )
    loss_fn = nn.CrossEntropyLoss()
    last_path = folder / f'{name}_last.pt'
    best_path = folder / f'{name}_best.pt'

    history = []
    best = -1.0
    start = 0

    if last_path.exists():
        ckpt = torch.load(last_path, map_location='cpu', weights_only=True)
        if ckpt['epochs'] != epochs:
            raise ValueError('Epoch setting changed. Use a new run name.')
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        start, best, history = ckpt['epoch'], ckpt['best'], ckpt['history']
        torch.set_rng_state(ckpt['torch_rng'])
        np_state = ckpt['numpy_rng']
        np.random.set_state((np_state[0], np.array(np_state[1], dtype=np.uint32),
                             np_state[2], np_state[3], np_state[4]))
        if device.type == 'cuda':
            torch.cuda.set_rng_state_all(ckpt['cuda_rng'])
        print('Resuming', name, 'from epoch', start)

    for epoch in range(start, epochs):
        model.train()
        total_loss = 0.0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(x), y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(y)

        loss_value = total_loss / len(loader.dataset)
        val_acc = accuracy(model, val_loader)
        history.append([epoch + 1, loss_value, val_acc])

        if val_acc > best:
            best = val_acc
            torch.save(model.state_dict(), best_path)

        np_state = np.random.get_state()
        torch.save({
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'epoch': epoch + 1,
            'epochs': epochs,
            'best': best,
            'history': history,
            'torch_rng': torch.get_rng_state(),
            'cuda_rng': torch.cuda.get_rng_state_all() if device.type == 'cuda' else [],
            'numpy_rng': (np_state[0], np_state[1].tolist(),
                          np_state[2], np_state[3], np_state[4])
        }, last_path)

        pd.DataFrame(history, columns=['epoch', 'train_loss', 'val_acc']).to_csv(
            folder / f'{name}_history.csv', index=False
        )
        print(f'{name}: {epoch+1:02}/{epochs} | loss {loss_value:.3f} | val {val_acc:.2f}%')

    return pd.DataFrame(history, columns=['epoch', 'train_loss', 'val_acc'])


def load_model(name, checkpoint='last'):
    path = folder / f'{name}_{checkpoint}.pt'
    saved = torch.load(path, map_location='cpu', weights_only=True)
    state = saved['model'] if checkpoint == 'last' else saved
    model = BigCNN().to(device)
    model.load_state_dict(state)
    return model

## 5. Source training — 40% dynamic label noise (60 epochs)

Train just once. Re-running the cell resumes from its saved epoch. This run's final checkpoint initializes all three transfer strategies.

In [ ]:
set_seed(seed)
source_model = BigCNN()
source_history = train_model(source_model, source_loader,
                             'source_noise40', source_epochs)

print('Final source loss:', round(source_history.train_loss.iloc[-1], 4))
print('Best clean validation:', round(source_history.val_acc.max(), 2), '%')

del source_model
torch.cuda.empty_cache()

## 6. Source health check (no training)

A source model that predicts only one class and has all-zero hidden representations is *collapsed*. The 80%-noise model showed exactly this problem. Check whether 40% corruption still allows meaningful representations before interpreting transfer results.

In [ ]:
source_model = load_model('source_noise40')
source_model.eval()
x, _ = next(iter(val_loader))

with torch.no_grad():
    features = source_model.conv_layers(x.to(device)).flatten(1)
    hidden = torch.relu(source_model.fc_layers[0](features))
    logits = source_model(x.to(device))

print('Mean feature std:', round(features.std(dim=0, unbiased=False).mean().item(), 6))
print('Zero hidden activations:', round(100 * (hidden == 0).float().mean().item(), 2), '%')
print('Logit variation:', round(logits.std(dim=0, unbiased=False).mean().item(), 6))

del source_model
torch.cuda.empty_cache()

## 7. Scratch BigCNN baseline (30 epochs)

For a fair comparison, all target methods must use the *same* BigCNN and target split. If the previous 80%-noise notebook's Scratch BigCNN and split files are present, this cell reuses that finished scratch run, saving time. Otherwise, it trains the scratch model here.

In [ ]:
old = root / 'transferguard_cifar10_results'
old_split = old / 'split_indices_seed42.npz'
old_best = old / 'scratch_big_best.pt'
old_hist = old / 'scratch_big_history.csv'
new_best = folder / 'scratch_big_best.pt'
new_hist = folder / 'scratch_big_history.csv'

same_split = False
if old_split.exists():
    with np.load(old_split) as saved_ids:
        same_split = all(np.array_equal(saved_ids[k], v) for k, v in [
            ('source', source_ids), ('target', target_ids),
            ('validation', val_ids), ('calibration', cal_ids)
        ])

if new_best.exists() and new_hist.exists() and len(pd.read_csv(new_hist)) >= target_epochs:
    print('Using completed scratch baseline in this notebook folder.')
elif (folder / 'scratch_big_last.pt').exists():
    set_seed(seed)
    scratch_model = BigCNN()
    train_model(scratch_model, target_loader, 'scratch_big', target_epochs)
    del scratch_model
    torch.cuda.empty_cache()
elif same_split and old_best.exists() and old_hist.exists() and len(pd.read_csv(old_hist)) == target_epochs:
    shutil.copy2(old_best, new_best)
    shutil.copy2(old_hist, new_hist)
    print('Reused your existing Scratch BigCNN checkpoint (matching split).')
else:
    set_seed(seed)
    scratch_model = BigCNN()
    train_model(scratch_model, target_loader, 'scratch_big', target_epochs)
    del scratch_model
    torch.cuda.empty_cache()

scratch_history = pd.read_csv(new_hist)
print('Scratch BigCNN best validation:', round(scratch_history.val_acc.max(), 2), '%')

## 8. Frozen transfer — reset **only the final classifier** (30 epochs)

The source backbone and first FC layer stay frozen. The last layer is reset and trained with clean target labels. This tests whether the fixed source representation is useful.

In [ ]:
set_seed(seed)
frozen = load_model('source_noise40')
for p in frozen.parameters():
    p.requires_grad = False
frozen.fc_layers[-1].reset_parameters()
for p in frozen.fc_layers[-1].parameters():
    p.requires_grad = True

frozen_history = train_model(frozen, target_loader,
                             'frozen_noise40_reset', target_epochs)

del frozen
torch.cuda.empty_cache()

## 9. Full fine-tuning — reset final classifier (30 epochs)

Every layer is trainable. This checks whether the 40%-noise source *initialization itself* helps or hurts when the backbone can adapt.

In [ ]:
set_seed(seed)
finetune = load_model('source_noise40')
finetune.fc_layers[-1].reset_parameters()
for p in finetune.parameters():
    p.requires_grad = True

finetune_history = train_model(finetune, target_loader,
                               'finetune_noise40_final_reset', target_epochs)

del finetune
torch.cuda.empty_cache()

## 10. Repaired fine-tuning — reset **both FC layers** (30 epochs)

This is the recovery strategy identified in your 80%-noise experiment. It tests whether retaining the convolutional layers but refreshing the classifier makes adaptation easier.

In [ ]:
set_seed(seed)
repaired = load_model('source_noise40')
repaired.fc_layers[0].reset_parameters()
repaired.fc_layers[-1].reset_parameters()
for p in repaired.parameters():
    p.requires_grad = True

repaired_history = train_model(repaired, target_loader,
                               'finetune_noise40_both_reset', target_epochs)

del repaired
torch.cuda.empty_cache()

## 11. Compare validation results

**Positive gap = the transfer method performs worse than the same-architecture Scratch BigCNN.** These are exploratory best-validation results from one seed, not final test conclusions. The plot and table are saved locally.

In [ ]:
names = [
    'scratch_big',
    'frozen_noise40_reset',
    'finetune_noise40_final_reset',
    'finetune_noise40_both_reset'
]

rows = []
plt.figure(figsize=(9, 5))

for name in names:
    history = pd.read_csv(folder / f'{name}_history.csv')
    best = history.val_acc.max()
    rows.append({'method': name, 'best_val_accuracy': best})
    plt.plot(history.epoch, history.val_acc, label=name)

results = pd.DataFrame(rows)
results.to_csv(folder / 'comparison_noise40.csv', index=False)
print(results.to_string(index=False))

scratch = float(results.loc[results.method == 'scratch_big',
                            'best_val_accuracy'].iloc[0])
for name in names[1:]:
    score = float(results.loc[results.method == name,
                              'best_val_accuracy'].iloc[0])
    print(f'Gap vs scratch ({name}): {scratch - score:+.2f} points')

plt.title('CIFAR-10 — 40% source-label noise')
plt.xlabel('Epoch')
plt.ylabel('Clean validation accuracy (%)')
plt.grid(alpha=0.3)
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(folder / 'comparison_noise40.png', dpi=150)
plt.show()

## 12. Sanity checks on the final models

If a transfer method stays near 10%, inspect its predictions and feature variation. A completely inactive model should be reported as **training collapse**, not automatically as ordinary negative transfer.

In [ ]:
for name in names:
    model = load_model(name, checkpoint='best')
    model.eval()
    counts = torch.zeros(10, dtype=torch.long)

    with torch.no_grad():
        for x, y in val_loader:
            pred = model(x.to(device)).argmax(1).cpu()
            counts += torch.bincount(pred, minlength=10)

        x, _ = next(iter(val_loader))
        features = model.conv_layers(x.to(device)).flatten(1)
        variation = features.std(dim=0, unbiased=False).mean().item()

    print('\n', name)
    print('Feature variation:', round(variation, 6))
    print('Predicted classes:', counts.tolist())

    del model
    torch.cuda.empty_cache()

## What to conclude after running

- **If frozen transfer loses but full fine-tuning succeeds:** the fixed source features are harmful, but adaptation can recover.
- **If full fine-tuning loses to scratch:** we have a *candidate* negative-transfer initialization effect; repeat with more seeds and check learning-rate sensitivity.
- **If resetting both FC layers recovers performance:** source-head initialization or gradient-flow pathology may explain the failure.
- **If all transfer methods beat scratch:** the 40%-noise source was still useful; report positive transfer instead of forcing a negative claim.

**Do not run the official CIFAR-10 test set to choose a method.** Next, add the clean-source control and develop the TransferGuard decision rule using the reserved calibration split. Only after freezing that protocol should we evaluate final test accuracy.

Output folder: `transferguard_noise40_results/` (checkpoints, CSV histories, comparison chart).